# Phishing detection
The models work on any classification task, not only founders. Here a Random Rule Forest learns which YES/NO questions flag phishing emails.

One key is enough, `OPENAI_API_KEY`: a chat model writes the questions and a smaller one answers them for every email, instead of Jev. A chat model answers YES or NO, and the elastic net weighs those answers.

In [ ]:
X = [
    "Account suspended. Verify your password at http://secure-paypa1.com within 24h.",
    "Hi team, the quarterly review moves to Thursday 10:00, room 4B. Agenda attached.",
    "You won a $500 gift card! Click here and enter your card details to claim it.",
    "Your order #48213 has shipped. Track it from the Orders page of your account.",
    "IT desk: your mailbox is full. Log in below with your password to keep mail.",
    "Reminder: your dentist appointment is on 14 March at 9:30. Reply C to cancel.",
    "Invoice overdue. Open the attached invoice.zip and enable macros to view it.",
    "Thanks for the draft. I left comments in the doc; can we talk tomorrow?",
    "Unusual activity on your account. Reply with your account number and PIN.",
    "Your statement is ready in the app. We never ask for your password by email.",
    "It's the CEO. Buy six $200 gift cards today and send me the codes. Tell no one.",
    "The library closes early on Friday. Borrowed items are now due on Monday.",
]
y = ["phishing", "legitimate"] * 6

In [ ]:
from think_reason_learn.rrf import RRF

rrf = RRF(
    task="Is this email a phishing attempt?",
    generator="openai:gpt-5-mini",  # writes the questions
    answerer="openai:gpt-5-nano",  # answers them for every email
    n_prompts=3,  # 10 prompts of 30 examples by default
    n_examples=6,
    cv=2,
    pos_label="phishing",
)
rrf.estimate_cost(X, y).cost_usd  # what fit would spend, before paying for anything

In [ ]:
rrf.fit(X, y)
rrf.rules_

In [ ]:
rrf.questions_.sort_values("f1", ascending=False)

In [ ]:
rrf.predict(
    [
        "Your parcel is held at customs. Pay the $2.99 fee at http://parcel-free.top.",
        "Lunch on Wednesday still works for me. Same place as last time?",
    ]
)

In [ ]:
rrf.usage_.by("role", "model")  # calls, tokens and dollars, per model